# 🤖 VLM Robot Agent
基于 LLaVA-1.5-7B 的机器人操作 Agent，BridgeData V2 数据集，Colab 免费 T4 可运行

In [ ]:
# ★ Cell 1: 修复 tokenizers 版本 bug（运行完后必须 Runtime → Restart session）
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'tokenizers==0.19.1', '--force-reinstall', '--no-deps'
], check=True)
import importlib
tok = importlib.import_module('tokenizers')
print(f'✅ tokenizers {tok.__version__} 安装完成')
print('⚠️  请立刻点击菜单：Runtime → Restart session，重启后跳过本 cell 继续执行')

In [ ]:
# Cell 2: 检查 GPU
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
import torch
print(f'CUDA: {torch.cuda.is_available()}')
print(f'GPU:  {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# Cell 3: 安装依赖
!pip install -q transformers==4.40.0 accelerate==0.29.3 bitsandbytes==0.43.1 \
    datasets Pillow einops huggingface_hub matplotlib
print('✅ 依赖安装完成')

In [ ]:
# Cell 4: 加载 LLaVA-1.5-7B 4bit 量化模型（约 4GB，5-10 分钟）
from transformers import LlavaForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
import torch

MODEL_ID = 'llava-hf/llava-1.5-7b-hf'

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
)

print('📥 下载模型中...')
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_cfg,
    device_map='auto',
    torch_dtype=torch.float16,
)
model.eval()
print(f'✅ 模型加载完成，VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

In [ ]:
# Cell 5: VLMRobotAgent 类定义
import json, re, torch
from PIL import Image

class VLMRobotAgent:
    """
    两阶段 VLM Robot Agent
    Stage 1 感知：理解图像中物体与位置
    Stage 2 规划：输出 JSON 结构化动作
    """
    ACTION_SPACE = ['move_to', 'grasp', 'release', 'push',
                    'pick_and_place', 'open', 'close', 'wait']

    def __init__(self, model, processor, max_new_tokens=200):
        self.model = model
        self.processor = processor
        self.max_new_tokens = max_new_tokens

    @torch.inference_mode()
    def _vlm(self, image, prompt_text):
        conversation = [{
            'role': 'user',
            'content': [
                {'type': 'image'},
                {'type': 'text', 'text': prompt_text},
            ]
        }]
        prompt = self.processor.apply_chat_template(
            conversation, add_generation_prompt=True
        )
        inputs = self.processor(
            images=image, text=prompt, return_tensors='pt'
        ).to(self.model.device, torch.float16)
        out = self.model.generate(
            **inputs,
            max_new_tokens=self.max_new_tokens,
            do_sample=False,
        )
        gen = out[0][inputs['input_ids'].shape[1]:]
        return self.processor.decode(gen, skip_special_tokens=True).strip()

    def perceive(self, image, instruction):
        prompt = (
            f'You are a robot arm camera. Task: {instruction}\n'
            'Describe: 1) all visible objects '
            '2) target object position (left/center/right, near/far) '
            '3) current scene state.'
        )
        return self._vlm(image, prompt)

    def plan(self, image, instruction, scene):
        actions = ', '.join(self.ACTION_SPACE)
        prompt = (
            f'Task: {instruction}\nScene: {scene}\n'
            f'Available actions: {actions}\n'
            'Output ONLY valid JSON: '
            '{"action":"...","target":"...","direction":"...","confidence":0.0}'
        )
        raw = self._vlm(image, prompt)
        m = re.search(r'\{[^{}]+\}', raw)
        try:
            return json.loads(m.group()) if m else {'action': 'wait', 'raw': raw}
        except Exception:
            return {'action': 'wait', 'raw': raw}

    def act(self, image, instruction):
        print(f'[Perceive] {instruction}')
        scene = self.perceive(image, instruction)
        print(f'  → {scene[:150]}')
        action = self.plan(image, instruction, scene)
        print(f'[Plan] → {action}\n')
        return {'instruction': instruction, 'scene': scene, 'action': action}


class CoTRobotAgent(VLMRobotAgent):
    """Chain-of-Thought 多步推理版，适合复杂长任务"""

    def act_cot(self, image, instruction):
        prompt = (
            f'Robot task: {instruction}\n\n'
            'Think step by step:\n'
            'Step 1: What objects are visible?\n'
            'Step 2: Which is the target object?\n'
            'Step 3: Where is it (left/center/right, near/far)?\n'
            'Step 4: Best action to execute?\n'
            'Step 5: Any obstacles or considerations?\n\n'
            'Final answer as JSON only: '
            '{"action":"...","target":"...","confidence":0.0}'
        )
        raw = self._vlm(image, prompt)
        print(f'CoT reasoning:\n{raw[:400]}\n')
        m = re.search(r'\{[^{}]+\}', raw)
        try:
            action = json.loads(m.group()) if m else {'action': 'wait'}
        except Exception:
            action = {'action': 'wait'}
        return {'instruction': instruction, 'cot': raw, 'action': action}


agent = VLMRobotAgent(model, processor)
cot_agent = CoTRobotAgent(model, processor, max_new_tokens=300)
print('✅ Agent ready')

In [ ]:
# Cell 6: 加载 BridgeData V2 样本（streaming 模式，不下载全集）
from datasets import load_dataset
import io
from PIL import Image

USE_BRIDGE = False
samples = []

try:
    print('📥 加载 BridgeData V2 streaming 样本...')
    ds = load_dataset(
        'jxu124/OpenX-Embodiment',
        name='bridge',
        split='train',
        streaming=True,
        trust_remote_code=True,
    )
    for i, s in enumerate(ds):
        if i >= 8: break
        samples.append(s)
    USE_BRIDGE = True
    print(f'✅ 加载 {len(samples)} 条真实样本')
except Exception as e:
    print(f'⚠️  BridgeData 加载失败: {e}')
    print('   自动切换到合成场景图像')


def get_bridge_image(sample):
    img = sample['images'][0]
    if isinstance(img, dict) and 'bytes' in img:
        return Image.open(io.BytesIO(img['bytes'])).convert('RGB')
    return img.convert('RGB')


# 合成场景（BridgeData 不可用时的备用）
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def make_scene(scene_type='kitchen'):
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.set_xlim(0, 10); ax.set_ylim(0, 7)
    ax.set_facecolor('#D2B48C')
    objs = {
        'kitchen': [
            dict(xy=(1.5,2), w=1.2, h=1.8, color='#FF4444', label='red cup'),
            dict(xy=(5,2.5), w=1.5, h=1.2, color='#4488FF', label='blue bowl'),
            dict(xy=(7.5,3), w=1.0, h=1.5, color='#44AA44', label='green bottle'),
        ],
        'drawer': [
            dict(xy=(2,1), w=6, h=2, color='#8B6914', label='drawer'),
            dict(xy=(3,4.5), w=1.0, h=0.8, color='#FF6600', label='orange object'),
        ],
    }
    for o in objs.get(scene_type, objs['kitchen']):
        ax.add_patch(patches.Rectangle(
            o['xy'], o['w'], o['h'],
            linewidth=2, edgecolor='black', facecolor=o['color'], alpha=0.85
        ))
        ax.text(o['xy'][0]+o['w']/2, o['xy'][1]+o['h']+0.2, o['label'],
                ha='center', fontsize=7, color='white',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.5))
    ax.axis('off')
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight', dpi=120,
                facecolor=ax.get_facecolor())
    buf.seek(0); plt.close(fig)
    return Image.open(buf).convert('RGB')


# 构建测试集
if USE_BRIDGE:
    test_cases = [
        {'image': get_bridge_image(s),
         'instruction': s.get('language_instruction', 'complete the task')}
        for s in samples[:5]
    ]
else:
    test_cases = [
        {'image': make_scene('kitchen'), 'instruction': 'Pick up the red cup and place it near the blue bowl.'},
        {'image': make_scene('kitchen'), 'instruction': 'Grasp the green bottle on the right side.'},
        {'image': make_scene('drawer'),  'instruction': 'Open the drawer and retrieve the orange object.'},
    ]

# 预览
fig, axes = plt.subplots(1, len(test_cases), figsize=(4*len(test_cases), 3))
if len(test_cases) == 1: axes = [axes]
for ax, tc in zip(axes, test_cases):
    ax.imshow(tc['image']); ax.axis('off')
    ax.set_title(tc['instruction'][:40], fontsize=7)
plt.tight_layout(); plt.show()
print(f'✅ {len(test_cases)} 个测试样本就绪')

In [ ]:
# Cell 7: 批量推理
import time

results = []
for i, tc in enumerate(test_cases):
    t0 = time.time()
    res = agent.act(tc['image'], tc['instruction'])
    res['time'] = round(time.time() - t0, 1)
    results.append(res)
    print(f'[{i+1}/{len(test_cases)}] 耗时 {res["time"]}s')

In [ ]:
# Cell 8: 可视化结果
import matplotlib.pyplot as plt

fig, axes = plt.subplots(len(results), 2, figsize=(12, 4*len(results)))
if len(results) == 1: axes = [axes]

for i, (tc, res) in enumerate(zip(test_cases, results)):
    axes[i][0].imshow(tc['image'])
    axes[i][0].set_title(f'Input #{i+1}', fontsize=10)
    axes[i][0].axis('off')

    plan = res['action']
    txt = (
        f"Instruction:\n{res['instruction']}\n\n"
        f"Scene (excerpt):\n{res['scene'][:200]}\n\n"
        f"Action Plan:\n"
        f"  action:     {plan.get('action','?')}\n"
        f"  target:     {plan.get('target','?')}\n"
        f"  direction:  {plan.get('direction','?')}\n"
        f"  confidence: {plan.get('confidence','?')}\n"
        f"  time: {res['time']}s"
    )
    axes[i][1].axis('off')
    axes[i][1].text(0.02, 0.95, txt, transform=axes[i][1].transAxes,
                    fontsize=9, va='top', fontfamily='monospace',
                    bbox=dict(boxstyle='round', facecolor='#111', alpha=0.8),
                    color='#e0e0ff')

plt.suptitle('VLM Robot Agent — 推理结果', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('results.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ 已保存 results.png')

In [ ]:
# Cell 9: 统计报告
from collections import Counter

actions = [r['action'].get('action', 'unknown') for r in results]
avg_t = sum(r['time'] for r in results) / len(results)

print('=' * 40)
print(f'  样本数:   {len(results)}')
print(f'  平均耗时: {avg_t:.1f}s')
print(f'  VRAM:    {torch.cuda.memory_allocated()/1e9:.1f} GB')
print('\n  动作分布:')
for a, n in Counter(actions).most_common():
    print(f'    {a:<18} {"█"*n} ({n})')
print('=' * 40)

In [ ]:
# Cell 10: CoT Agent 示例（复杂任务）
tc = test_cases[-1]
print(f'CoT Agent 推理: {tc["instruction"]}\n')
cot_result = cot_agent.act_cot(tc['image'], tc['instruction'])
print('最终动作:')
print(json.dumps(cot_result['action'], indent=2, ensure_ascii=False))